# Dự án Capstone "THE PRICE IS RIGHT"

Tuần này - xây dựng một mô hình dự đoán giá của một sản phẩm từ mô tả, dựa trên dữ liệu được scrape từ Amazon

Một mô hình có thể ước tính giá của một sản phẩm từ mô tả của nó.

# Trình tự

NGÀY 1: Thu thập và tinh chỉnh dữ liệu  
NGÀY 2: Tiền xử lý dữ liệu  
NGÀY 3: Đánh giá, baseline, Machine Learning truyền thống  
NGÀY 4: Deep Learning và LLMs  
NGÀY 5: Fine-tuning một mô hình Frontier  

## NGÀY 2: Tiền xử lý dữ liệu

Hôm nay chúng ta sẽ viết lại các sản phẩm theo một định dạng chuẩn.  
LLM rất giỏi ở việc này!

## Tóm tắt quy trình của notebook

1. Tải dữ liệu thô từ Hugging Face Hub.
2. Chọn dataset lite hoặc full tùy mức ngân sách và tốc độ xử lý.
3. Gắn ID cho từng item để dễ theo dõi.
4. Dùng prompt hệ thống để yêu cầu mô hình viết lại sản phẩm theo cấu trúc chuẩn.
5. Tạo file JSONL để gửi batch lên Groq.
6. Chờ xử lý hàng loạt và lưu kết quả lại vào dataset.
7. Xóa các trường dư thừa để chuẩn bị push lên Hub.
8. Đẩy dataset đã tiền xử lý lên Hugging Face để dùng cho các notebook sau.

## Ý nghĩa chính của notebook

Notebook này dạy cách biến dữ liệu thô chưa sạch thành dữ liệu đầu vào có cấu trúc, có thể dùng để huấn luyện mô hình.  
Nó also nhấn mạnh một kỹ thuật rất mạnh: dùng LLM để viết lại dữ liệu theo một định dạng nhất quán, thay vì làm thủ công từng item.

## Mục tiêu cuối cùng

Mục tiêu cuối cùng là tạo ra một tập dữ liệu sản phẩm đã được tóm tắt và chuẩn hóa, có thể dùng cho các bước học máy sau này, đặc biệt là dự đoán giá và tìm hiểu loại dữ liệu nào thực sự hiệu quả cho mô hình.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Giá trị kinh doanh của việc tiền xử lý / viết lại dữ liệu</h2>
            <span style="color:#181;">LLM đã làm cho việc thực hiện một thứ từng được coi là không thể trong vài năm trở lại đây trở nên đơn giản.
            Cách tiếp cận này có thể được áp dụng cho hầu hết mọi lĩnh vực kinh doanh, và nó tương tự như các kỹ thuật nâng cao
            mà chúng ta đã dùng ở Tuần 5.</span>
        </td>
    </tr>
</table>


In [ ]:
# Import các thư viện cần thiết cho việc gọi mô hình và xử lý dữ liệu
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

# Tải biến môi trường từ file .env
load_dotenv(override=True)


# Ô kế tiếp là nơi bạn chọn Dataset

Sử dụng `LITE_MODE = True` cho phiên bản miễn phí, nhanh với kích thước dữ liệu huấn luyện là 20,000

Sử dụng `LITE_MODE = False` cho phiên bản đầy đủ, mạnh hơn với kích thước dữ liệu huấn luyện là 800,000

## Cho bài lab này

Bạn có thể bỏ qua hoàn toàn và tải dataset từ HuggingFace: $0

Bạn có thể chạy tiền xử lý cho dataset lite: dưới $1

Bạn có thể chạy tiền xử lý cho dataset đầy đủ: $30


In [ ]:
# Chế độ lite giúp chạy nhanh và ít tốn tiền hơn
# Đặt True để dùng dataset lite (20,000 mẫu)
# Đặt False để dùng dataset đầy đủ (800,000 mẫu)
LITE_MODE = True


In [ ]:
# Chọn tên dataset theo chế độ đang dùng
# Nếu LITE_MODE=True -> dùng data lite, nếu False -> dùng data full
username = "cuong-phan"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

# Tải train, val, test từ Hugging Face Hub
train, val, test = Item.from_hub(dataset)

# Gộp tất cả thành một danh sách items để xử lý
items = train + val + test

print(f"Đã tải {len(items):,} item")
print(items[0])


In [ ]:
items[2].id

In [ ]:
# Gán mã định danh cho từng sản phẩm để theo dõi dữ liệu dễ hơn
# Sau này khi xử lý batch, ta cần biết item nào tương ứng với output nào
for index, item in enumerate(items):
    item.id = index


In [ ]:
# Prompt hệ thống yêu cầu mô hình trả về mô tả sản phẩm theo một định dạng chuẩn
# Mục tiêu: mỗi item phải có title, category, brand, description và details
# LLM chỉ được trả lời theo format này; không được include part numbers
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""


In [ ]:
# In ra nội dung thô của sản phẩm đầu tiên để xem dữ liệu ban đầu như thế nào
print(items[0].full)


In [ ]:
# Gọi mô hình qua LiteLLM để kiểm tra xem prompt có hoạt động không
# Đây là một test nhanh trên 1 item đầu tiên
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

# In ra nội dung trả về của mô hình
print(response.choices[0].message.content)
print()

# In ra thống kê token và chi phí để theo dõi
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [ ]:
# Test tương tự nhưng bằng mô hình chạy local trên máy với Ollama
# Mục tiêu: kiểm tra xem có thể dùng mô hình local thay vì API cloud hay không
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [ ]:
# Tên mô hình sẽ dùng trong file JSONL để gửi batch cho Groq
# Đây là model được dùng để rewrite từng sản phẩm
MODEL = "openai/gpt-oss-20b"


In [ ]:
# Hàm này tạo ra 1 dòng JSON cho 1 item, đúng format mà Groq Batch API yêu cầu
# Mỗi dòng đại diện cho một request POST tới /v1/chat/completions

def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)


In [ ]:
# Xem 1 item đầu tiên để xác nhận cấu trúc dữ liệu
items[0]


In [ ]:
# Chuyển item đầu tiên thành định dạng JSONL để kiểm tra xem cú pháp đã đúng chưa
make_jsonl(items[0])


In [ ]:
# Hàm này tạo file JSONL chứa nhiều request cho khoảng [start, end)
# Ví dụ: make_file(0, 1000, "jsonl/0_1000.jsonl") sẽ tạo 1000 request

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")


In [ ]:
# Tạo file batch cho 1000 item đầu tiên
# Đây là đoạn dữ liệu thử nghiệm trước khi chạy batch lớn hơn
make_file(0, 1000, "jsonl/0_1000.jsonl")


In [ ]:
# Import thư viện Groq SDK để upload file batch lên Groq
import os
from groq import Groq

# Khởi tạo client với API key từ biến môi trường GROQ_API_KEY
# Nếu không có key, bạn sẽ không thể gửi request lên Groq
groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))


In [ ]:
# Upload file JSONL lên Groq để tạo input file cho batch
# `purpose="batch"` nói với Groq rằng file này sẽ được dùng cho job batch
with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response


In [ ]:
# Lưu lại ID của file upload, vì ta cần dùng nó để tạo batch job
file_id = response.id
file_id


In [ ]:
# Khởi tạo batch job trên Groq với endpoint chat/completions
# `completion_window="24h"` nghĩa là job sẽ chạy trong vòng 24 giờ
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response


In [ ]:
# Kiểm tra trạng thái batch hiện tại bằng ID của nó
result = groq.batches.retrieve(response.id)
result


In [ ]:
# Khi batch hoàn tất, lấy file kết quả và lưu về local
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")


In [ ]:
# Đọc từng dòng kết quả batch và gán summary vào item tương ứng
# Mỗi response chứa `custom_id`, nên ta có thể map đúng với item.id
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [ ]:
# Kiểm tra dữ liệu gốc trước khi rewrite
print(items[0].full)


In [ ]:
# In ra summary của item thứ 1000 để xác nhận đã có dữ liệu sau batch
print(items[1000].summary)


## Tôi đã đặt đúng logic này vào class Batch

- Chia danh sách item thành từng nhóm 1,000
- Khởi tạo batch cho từng nhóm
- Giám sát trạng thái và thu thập kết quả khi hoàn tất

## Chi phí

Dùng Groq, với tôi thì chi phí khoảng dưới $1 cho dataset Lite và dưới $30 cho dataset lớn

Nhưng bạn không cần phải trả gì cả! Ở lab tiếp theo, bạn có thể tải kết quả đã tiền xử lý sẵn của tôi


In [ ]:
# Gọi các bước đã được encapsulate trong class Batch
# Batch.create(...) sẽ chia dữ liệu và tạo request
# Batch.run() thực thi các batch
# Batch.fetch() lấy kết quả khi hoàn tất
Batch.create(items, LITE_MODE)


In [ ]:
# Chạy tất cả batch đã được tạo
Batch.run()


In [ ]:
# Lấy kết quả từ Groq sau khi batch chạy xong
Batch.fetch()


In [ ]:
# Tìm các item chưa được gán summary
# Nếu list này rỗng, nghĩa là mọi item đã xử lý xong
for index, item in enumerate(items):
    if not item.summary:
        print(index)


In [ ]:
# In ra summary của một item ngẫu nhiên để kiểm tra chất lượng output
print(items[10234].summary)


In [ ]:
# Xóa các trường không cần thiết trước khi push lên Hugging Face
# Ta muốn chỉ giữ các field có giá trị thật sự để train model
for item in items:
    item.full = None
    item.id = None


## Đẩy dataset cuối cùng lên Hub

Nếu chạy ở chế độ lite, ta chỉ push dataset lite

Nếu chạy ở chế độ full, ta sẽ push cả dataset full (để sau này bạn có thể quay lại dùng lite nếu muốn)

## Tóm tắt quy trình của notebook

1. Tải dữ liệu thô từ Hugging Face.
2. Chia thành các chunk để xử lý hàng loạt với LLM.
3. Gửi request JSONL lên Groq batch API.
4. Thu nhận kết quả và ghi summary vào từng item.
5. Xóa các trường dư thừa để giảm kích thước dữ liệu và giữ thông tin cần thiết.
6. Đẩy dataset đã tiền xử lý lên Hub để làm tài nguyên huấn luyện sau này.

## Ý nghĩa chính của notebook

Notebook này cho thấy cách làm việc với dữ liệu lớn theo hướng production: tách batch, dùng API xử lý hàng loạt, rồi lưu kết quả ở dạng dataset chuẩn hóa. Đây là bước trung gian rất quan trọng trước khi xây mô hình dự đoán giá.

## Mục tiêu cuối cùng

Mục tiêu cuối cùng là biến tập dữ liệu Amazon raw thành tập dữ liệu sạch, đồng nhất và dễ sử dụng để huấn luyện mô hình price prediction sau này.


In [ ]:
# Đặt tên dataset trên Hugging Face Hub
# Nếu chạy chế độ lite -> chỉ upload lite dataset
# Nếu chạy chế độ full -> upload cả full và lite dữ liệu phụ để tiện dùng sau này
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)


## Và đây là kết quả!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full

> Bản chất của quy trình này là: dữ liệu thô -> chuẩn hóa bằng LLM -> lưu trên Hub -> dùng cho các notebook sau.
